In [1]:
import sys
import filepath
import pdb

In [2]:
from typing import Dict
from PyQt5.QtWidgets import (
    QWidget, QApplication, QMainWindow,
    QDockWidget, QTextEdit, QPlainTextEdit
)
from PyQt5.QtCore import QObject, QEvent,Qt
from panda3d.core import (
    loadPrcFileData , NodePath
)
from qpanda3d import (
    QShowBaseMultiView, QPanda3DWidget, QControlMultiView,
    Synchronizer
)

from config.style import styleSheet
from qtutil.event import *
from util.log import *
from ui.abstract_ui import AbstractGUI
from ui.qtui.qconsole import *
from ui.qtui.qlogger import *

In [3]:
from ui.qtui.main_window import MultiViewQtGUI
from art.basic import (
    create_cube_node, create_sphere_node,
uv_curve_surface,create_colored_cube_node
)

In [4]:
from art.procedural_art.perlin_landmap import fractal_perlin_custom_lac
from util.geometry import format_normal
import torch
noise = fractal_perlin_custom_lac(
    (64,64),
    (4,4),
    lacunarity_list=[1.5, 2.2, 2.8, 1.7, 3.0]
)
z = noise * 100
z[-1] = 0
z[0] = 0
z[:,-1] = 0
z[:,0] = 0
x = torch.ones_like(z)
y = torch.ones_like(z)
# x = x * torch.lin
x = x * torch.linspace(-256, 255, 64).unsqueeze(0)
y = y * torch.linspace(-256, 255, 64).unsqueeze(1)
noise.median() * 100
xyz = torch.concat([
    x.unsqueeze(-1),
    y.unsqueeze(-1),
    z.unsqueeze(-1)
],dim=-1) 
landscape = uv_curve_surface(
    "land", xyz, False,False,
    vformat=format_normal
)

/mnt/D/packages/miniconda3/envs/game/lib/python3.12/site-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


In [5]:
from direct.showbase.ShowBase import ShowBase
from panda3d.core import (
    AmbientLight,
    DirectionalLight,
    PointLight,
    Shader,
    Material,
    Vec3,
    Vec4,
GeomNode
)
from panda3d_game.app import ContextShowBase
# Shader
# Application Class
# =================
normals = []

class ShaderDemo(ContextShowBase):
    def __init__(self):
        # Call the base constructor
        ContextShowBase.__init__(self)
        # self.sphere = create_sphere_node("sphere", lat_res=24, lon_res=24,vformat=format_normal)
        # self.sphere_np = NodePath(self.sphere)
        self.land = GeomNode("land")
        self.land.addGeom(landscape)
        self.land_np = NodePath(self.land)
        self.land_np.reparent_to(self.render)
        # self.indicator = create_sphere_node("sphere", lat_res=4, lon_res=24)
        # self.indicator_np = NodePath(self.indicator)
        # self.indicator_np.set_scale(.1)
        # self.indicator_np.reparent_to(self.render)


        # 创建材质

        # mat = Material()
        # mat.setAmbient((1, 1, 1, 1))# 环境光反射
        # mat.setSpecular((0.5, 0.5, 0.5, 1))# 镜面光颜色
        # mat.setShininess(20)# 镜面高光大小/锐利程度
        # mat.setDiffuse((0.8, 0.8, 0.8, 1))  # RGB 越接近 1 越亮/浅  # 漫反射
        
        
        # self.sphere_np.setMaterial(mat)
        

        self.ambient_light = self.render.attach_new_node(AmbientLight("AmbientLight"))
        # self.ambient_light.node().set_color(Vec4(.2, .2, .2, 1))
        self.ambient_light.node().set_color(Vec4(.2, .2, .2, 1))
        self.render.set_light(self.ambient_light)
        
        self.sun = self.render.attach_new_node(DirectionalLight("Sun"))
        self.sun.set_hpr(45, -45, 0)
        self.render.set_light(self.sun)
        
        # self.green_light = self.render.attach_new_node(PointLight("GreenLight"))
        # self.green_light.node().set_color(Vec4(0, 1, 0, 1))
        # self.green_light.node().set_attenuation(Vec3(1, .1, .5))
        # self.green_light.node().set_attenuation(Vec3(1, 0.1, 0.05))
        # self.green_light.node().set_attenuation(Vec3(1,0,0))
        # self.green_light.set_pos(0, 3, 100)
        # self.render.set_light(self.green_light)
        # # self.indicator_np.set_pos(0,3,0)
        

        # geom = self.sphere_np.node().getGeom(0)
        # vdata = geom.getVertexData()
        # print(vdata.hasColumn("normal"))
        # from panda3d.core import GeomVertexReader
        # normal_reader = GeomVertexReader(vdata, "normal")
        # while not normal_reader.isAtEnd():
        #     n = normal_reader.getData3f()
        #     normals.append(n)
        # print(normals[:10])
        # self.terrain_shader = Shader.make(
        #     Shader.SL_GLSL,
        #     terrain_vert,
        #     terrain_frag
        # )
        # print(self.terrain_shader)
        # self.land_np.set_shader(self.terrain_shader)
        # self.land_np.set_shader_input("texScale0", Vec2(.1, .1))
        
        # self.land_np.set_texture(tdirt)
        # self.water = WaterPlane(
        #     Vec3(0, 0, 50),
        #     scale=Vec3(256, 256, 1)
        # )
        # self.water

        # self.sphere_shader = Shader.make(
        #     Shader.SL_GLSL,
        #     sphere_vert_diffuse,
        #     # sphere_frag_diffuse
        #     sphere_frag_0
        #     # specular_vert,
        #     # specular_frag
        #     # sphere_vert,
        #     # sphere_frag_ambient
        # )
        # self.sphere_np.set_pos(0, 5, 0)
        # self.sphere_np.set_shader(self.sphere_shader)
        # self.sphere_np.reparent_to(self.render)
        # Initialize water material if necessary
        # if self.water_mat is None:
        
        # self.water.set_shader(self.water_shader)

        # self.water.set_material(self.water_mat)

    def pause_switch(self, *args,**kwargs):
        pass


In [6]:
from qpanda3d import  QPanda3DWidget, Synchronizer
from demos.physics_room import PhyscRoomConsole
class ShaderControl(ShaderDemo, QControlMultiView):
    def __init__(self):
        QControlMultiView.__init__(self)
        ShaderDemo.__init__(self)
    # def __init__(self, isQt = True):
    #     import pdb
    #     QControl.__init__(self)
    #     ShaderDemo.__init__(self)
    #     self.isQt = isQt
    #     if self.isQt:
    #         self.startQt()
from demos.physics_room import PhyscRoomConsole
from ui.qtui import *

# class ShaderGame(RawQtGUI):
#     def get_game(self):
#         return ShaderControl()

#     def get_console(self):
#         return PhyscRoomConsole(showbase=self.panda3d)

In [7]:
class TestUI(MultiViewQtGUI):
    def get_game(self):
        return ShaderControl()

    def get_console(self):
        return PhyscRoomConsole(showbase=self.panda3d)
  

In [8]:
if __name__ == '__main__':
    # torch.set_printoptions(precision=16, sci_mode=False)
    import sys
    app = QApplication(sys.argv)
    window = TestUI()
    window.show()
    sys.exit(app.exec_())

Known pipe types:
  glxGraphicsPipe
(all display modules loaded.)
/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory
:audio(error): Couldn't open default OpenAL device
:audio(error): OpenALAudioManager: No open device or context
:audio(error):   OpenALAudioManager is not valid, will use NullAudioManager
:audio(error): Couldn't open default OpenAL device
:audio(error): OpenALAudioManager: No open device or context
:audio(error):   OpenALAudioManager is not valid, will use NullAudioManager


cin
cout
key down escape
send button escape 16777216
key up escape
send button-up escape-up 16777216
cin
key down b
send button b 66
key up b
send button-up b-up 66
key down shift
send button shift 16777248
key down shift-:
send button shift-: 58
key up shift-:
send button-up shift-:-up 58
key up shift
send button-up shift-up 16777248
key down a
send button a 65
key down s
send button s 83
key down d
send button d 68
key up a
send button-up a-up 65
key up s
send button-up s-up 83
key down f
send button f 70
key up d
send button-up d-up 68
key up f
send button-up f-up 70
key down shift
send button shift 16777248
key down shift-:
send button shift-: 58
key up shift-:
send button-up shift-:-up 58
key up shift
send button-up shift-up 16777248
key down shift
send button shift 16777248
key down shift-:
send button shift-: 58
key up shift-:
send button-up shift-:-up 58
key up shift
send button-up shift-up 16777248
key down shift
send button shift 16777248
key down shift-:
send button shift-: 

SystemExit: 0

/mnt/D/packages/miniconda3/envs/game/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
window

In [1]:
# k = f"{get_panda_key_modifiers_prefix(evt)}{QPanda3D_Key_translation[key]}"
from QPanda3D.QPanda3D_Buttons_Translation import QPanda3D_Button_translation
from QPanda3D.QPanda3D_Keys_Translation import QPanda3D_Key_translation
from QPanda3D.QPanda3D_Modifiers_Translation import QPanda3D_Modifier_translation
key = 58
k = QPanda3D_Key_translation[key]

In [2]:
k

'unknown'

In [1]:
# from PyQt5 import Qt
from PyQt5.QtCore import Qt

In [2]:
Qt.Key_Semicolon

59

In [3]:
Qt.Key_Colon

58

In [6]:
Qt.Key_Escape

16777216

In [7]:
Qt.Key_Shift

16777248

In [11]:
QPanda3D_Key_translation[Qt.Key_Shift]

'shift'

In [10]:
from qpanda3d.qpanda3d_keys_translations import QPanda3D_Key_translation